## Lecture 5: Textual analysis

1. How to access 10-K filings
2. Loading a sample of 10-K filings
3. Text mining basics
4. Topic models

Course book:
Chapter 10: Natural Language Processing in Finance, Economics, and Accounting

Mikko Ranta

Where to find 10-k data:
https://sraf.nd.edu/sec-edgar-data/cleaned-10x-files/

In [ ]:
import zipfile
import re
from pathlib import Path
import pandas as pd

import nltk
from nltk import word_tokenize, pos_tag
from nltk.corpus import stopwords
from nltk.corpus.reader.wordnet import NOUN, VERB, ADJ, ADV
from nltk.stem import WordNetLemmatizer
from gensim.utils import simple_preprocess
from collections import Counter

from gensim.models.phrases import Phrases, Phraser

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
#Read the document and filename into variables. 
#This time the filename is not important but in case your data includes multiple companies it can be an important identifier

documents = []
filenames = []


#This loop reads all txt files contained in a zip file and saves them into a variable documents
with zipfile.ZipFile("/home/jovyan/shared/Data/Walmart 10-K.zip") as z:

    for file in z.namelist():

        if file.lower().endswith(".txt"):

            text = z.read(file).decode(
                "utf-8",
                errors="ignore"
            )

            documents.append(text)
            filenames.append(file)

In [ ]:
documents

In [ ]:
filenames

## Quick analysis

Extract document length for each document and plot it yearly.

Get the count of a specific word for each document.

If we also want to collect the most common words and more specific details for each document we will have to preprocess the data first. Otherwise the most common words will be non-informative words such and "the" and "and".

In [ ]:

word_counts = []

for i, doc in enumerate(documents):

    # Extract words (letters only)
    words = re.findall(r'\b[a-zA-Z]+\b', doc.lower())

    word_counts.append({
        "filename": filenames[i],
        "word_count": len(words)
    })

# Create dataframe
word_count_df = pd.DataFrame(word_counts)

# Display results
print(word_count_df)

In [ ]:
#Add the reporting year to our data
#This only works if all filenames end with the year + 3-digit file format
year_list = [name[-8:-4] for name in filenames]

word_count_df["year"] = year_list

print(word_count_df.head())

In [ ]:
#Sort the data in chronological order
word_count_df = word_count_df.sort_values(by='year')

#Plot the number of words per year
plt.figure(figsize=(10, 5))
word_count_df.plot(x= "year", y="word_count", kind="line", marker="o")

plt.title("Total Words per Year")
plt.xlabel("Year")
plt.ylabel("Number of Words")

plt.grid(True)
plt.show()

In [ ]:
#Counting a specific word
word = "sustainability"

word_count_df[f"word_count_{word}"] = [
    len(re.findall(
        r'\b' + re.escape(word) + r'\b',
        doc,
        flags=re.IGNORECASE
    ))
    for doc in documents
]

word_count_df

In [ ]:
#Plot the number of specific words per year
plt.figure(figsize=(10, 5))
word_count_df.plot(x= "year", y=f"word_count_{word}", kind="line", marker="o")

plt.title("Total Words per Year")
plt.xlabel("Year")
plt.ylabel("Number of Words")

plt.grid(True)
plt.show()

## Data pre-processing

Depending on what you plan to do, textual data needs to be processed to make analysis easier and results meaningful when using so-to-speak older techniques and not modern language models. The benefit of the older models is that they require less computation power and are accurate when used for their specific purpose, for example, word counts.

This means for example removing commonly occurring words (the, and, in, etc.), because if we want to learn about the substance of the text, these words don't bring us additional information. We also remove punctuation, numbers and other unnecessary markdown. We can also create word pairs in the pre-processing stage to analyse which words commonly occur together.


From the [coursebook](https://mranta-ai.github.io/data-science-in-finance/nlp-in-finance-econ-and-ba/):
### Financial text preprocessing presents unique challenges compared to standard computational linguistics:

* Numbers, Percentages, and Currencies: Monetary amounts, basis points (bps), and balance sheet items must be parsed or filtered depending on the downstream task.

* Boilerplate & Legal Disclaimer Scrubbing: SEC 10-Ks and FOMC releases contain standardized legal boilerplate (“forward-looking statement safe harbor”) that drowns out economic signal.

* Collocation & Compound Financial N-Grams: Words like federal and funds or interest and rate carry distinct meanings when combined as bigrams (federal_funds, interest_rate).

* Part-of-Speech (POS) Filtering & Lemmatization: Isolating content-bearing words (Nouns, Verbs, Adjectives) while reducing inflected variants to their root dictionary form (lemma).

Stop-words include common words like the, and, in, on etc. that don't bear content or contextual information. In addition we can add our own stop-words to the list. This time we add a list of common words for 10-K filings.

In [ ]:
#load a stop-word list
stop_words = set(stopwords.words("english"))
stop_words

#Walmart- and financial specific stop-words
add_stop_words = ['company',
 'fiscal',
 'include',
 'financial',
 'walmart',
 'sale',
 'operation',
 'u.s.',
 'january',
 'store',
 'net',
 'tax',
 'income',
 'result',
 'statement',
 'business',
 'year',
 'club',
 'operate',
 'information',
 'wmt',
 'change',
 'cash',
 'certain',
 'segment',
 'report',
 'rate',
 'cost',
 'market',
 'us-gaap',
 'value',
 'customer',
 'share',
 'international',
 'amount',
 'asset',
 'total',
 'item',
 'loss',
 'affect',
 'form',
 'file',
 'increase',
 'unit',
 'stock',
 'provide',
 'end',
 'date',
 'use',
 'consolidated',
 'associate',
 'service',
 'retail',
 'merchandise',
 'period',
 'officer',
 'annual',
 'law',
 'sam',
 'country',
 'relate',
 'well',
 'risk',
 'performance',
 'security',
 'state',
 'product',
 'executive',
 'adversely',
 'liability',
 'reference',
 'incorporate',
 'also',
 'price',
 'matter',
 'interest',
 'control',
 'debt',
 'exchange',
 'management',
 'exhibit',
 'fair',
 'due',
 'factor',
 'operating',
 'ecommerce',
 'new',
 'require',
 'lease',
 'plan',
 'impact',
 'expense',
 'purchase',
 'distribution',
 'make',
 'regulation',
 'inc.',
 'payment',
 'note',
 'facility',
 'gaap',
 'member']

stop_words = stop_words.union(add_stop_words)

Regular expressions are used to match a specific pattern in text. [https://regex101.com/](https://regex101.com/) can be used to decode regular expressions.

In [ ]:
#Preprocessing function

#Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

#Defining the function does not execute it
def preprocess(text):

    #remove yyyy-dd-mm or yyyy-mm-dd format dates and years
    text = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", " ", text)
    text = re.sub(r"\b(?:19|20)\d{2}\b", " ", text)

    #simple_preprocess is the gensim function with built-in pre-processing steps
    tokens = simple_preprocess(text, deacc=True, min_len=3)

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return tokens

In [ ]:
#Now execute the function

processed_docs = [
    preprocess(doc)
    for doc in documents
]

In [ ]:
print(processed_docs[0])

Add the reporting year to our document data and analyse further.

In [ ]:
#Dataframe to make visualization more convenient
#Combine the filename variable, document variable and year variable

df = pd.DataFrame(
    list(zip(filenames, processed_docs, year_list)),
    columns=["filenames", "tokens", "year"]
)

df.head()

In [ ]:
#Create one list containing all words in our set of documents
all_tokens = [
    token
    for doc in df["tokens"]
    for token in doc
]

#Count the number of occurrences for each word
word_counts = Counter(all_tokens)

#Get the 20 most common words and create a dataframe
top_words = pd.DataFrame(
    word_counts.most_common(20),
    columns=["word", "count"]
)

plt.figure(figsize=(12, 6))
plt.bar(top_words["word"], top_words["count"])

plt.xticks(rotation=45, ha="right")
plt.title("Top 20 Words")
plt.tight_layout()
plt.show()

In [ ]:
#Most common words per each year
#note the years are stored in string type
year = "2019"

#Only get words for the specified year
tokens = [
    token
    for doc in df.loc[df["year"] == year, "tokens"]
    for token in doc
]

top = Counter(tokens).most_common(15)

words = [w for w, c in top]
counts = [c for w, c in top]

plt.figure(figsize=(10, 5))
plt.bar(words, counts)

plt.title(f"Top Words in {year}")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Topic analysis

### Latent Dirichlet allocation

Latent Dirichlet allocation (LDA) is a mixed-membership multinomial clustering model (Blei, Ng, and Jordan 2003) that generalizes naive Bayes. Using the topic and document terminology common in discussions of LDA, each document is modeled as having a mixture of topics, with each word drawn from a topic based on the mixing proportions. (From [https://mc-stan.org/docs/stan-users-guide/clustering.html#latent-dirichlet-allocation](https://mc-stan.org/docs/stan-users-guide/clustering.html#latent-dirichlet-allocation))

In [ ]:
# Extract bigrams

bigram_model = Phrases(
    processed_docs,
    #Only bigrams that exist at least 5 times in our collection of texts
    min_count=5,
    threshold=10
)

bigram_phraser = Phraser(bigram_model)

bigram_docs = [
    bigram_phraser[doc]
    for doc in processed_docs
]

In [ ]:
#Get most common bigrams
#Score does not mean the number of times the bigram occurs

for phrase, score in sorted(
    bigram_model.export_phrases().items(),
    key=lambda x: x[1],
    reverse=True
)[:30]:

    print(phrase, score)

In [ ]:
from gensim import corpora
from gensim.models import LdaModel

In [ ]:
# Create dictionary
dictionary = corpora.Dictionary(bigram_docs)

# Optional: remove very rare and very common terms
dictionary.filter_extremes(
    no_below=2,    # appears in at least 2 docs 
    no_above=0.9   # does not appear in more than 80% of docs
)

# Create bag-of-words corpus
corpus = [
    dictionary.doc2bow(doc)
    for doc in bigram_docs
]

In [ ]:
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=10,
    iterations=100,
    alpha="symmetric", #Symmetric assumes that the topics hold equal importance
    eta="auto"
)

In [ ]:
#Check the topics

for topic_id in range(lda_model.num_topics):

    print(f"\nTopic {topic_id}")

    terms = lda_model.show_topic(
        topic_id,
        topn=10
    )

    for word, weight in terms:
        print(f"{word:<25} {weight:.4f}")

In [ ]:
dominant_topics = []

for doc_bow in corpus:

    topic_probs = lda_model.get_document_topics(
        doc_bow
    )

    dominant_topic = max(
        topic_probs,
        key=lambda x: x[1]
    )[0]

    dominant_topics.append(dominant_topic)

df["topic"] = dominant_topics

In [ ]:
df

In [ ]:
topic_year = pd.crosstab(
    df["year"],
    df["topic"]
)

topic_year.plot(
    figsize=(12, 6),
    marker="o"
)

## Task

Try changing the model and dictionary parameters and see how it affects the topics